## My Capstone Plan

**Domain:** Agentic AI Hands-On Course Assistant

**User:** B.Tech 4th-year students who need help understanding session topics, concepts, code patterns, warnings, and submission requirements from the 13-day Agentic AI course taught by Dr. Kanthi Kiran Sirra.

**Success looks like:** The agent correctly answers concept questions from any of the 13 course sessions, admits when it does not have information, never fabricates library versions or code patterns, and maintains context within a study session (e.g. remembers earlier questions in the same session).

**Tool I will add:** `get_current_datetime` — returns today's date and day so the agent can answer questions like "which day of the course is today?" without hallucinating.

**Deployment choice:** Streamlit UI — students open it in a browser during study sessions.

---
## 0. Setup

In [1]:
# ============================================================
# LOCAL SETUP — run once in your terminal before opening this notebook
# ============================================================
# pip install -r requirements.txt
#
# Create a .env file in the project root with:
#   GROQ_API_KEY=your_key_here

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key : {'OK — loaded' if len(groq_key) > 10 else 'MISSING — add to .env'}")
print(f"LangGraph    : {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM          : {r.content}")

c:\Users\KIIT0001\OneDrive\Desktop\Agentic_AI\Capstone Project Joy\.capstonejoy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq API Key : OK — loaded
LangGraph    : 1.1.6
LLM          : Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [3]:
# ── Import shared knowledge base from agent.py ────────────

from agent import DOCUMENTS, build_knowledge_base

print("Loading embedding model (downloads ~90 MB on first run)...")
embedder, collection = build_knowledge_base()

print(f"\nKnowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   [{d['id']}] {d['topic']}")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5594.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Knowledge base ready: 13 documents
   [doc_001] Day 1 — Python & API Foundations
   [doc_002] Day 2 — Tools and Function Calling
   [doc_003] Day 3 — LangChain Chains and Prompt Templates
   [doc_004] Day 4 — Retrieval-Augmented Generation (RAG) and ChromaDB
   [doc_005] Day 5 — LangGraph StateGraph and Agent Architecture
   [doc_006] Day 6 — Memory and Conversation History
   [doc_007] Day 7 — Self-Reflection and Eval Node
   [doc_008] Day 8 — Multi-Tool Agents and Router Design
   [doc_009] Day 9 — FastAPI and Production Packaging
   [doc_010] Day 10 — Streamlit Deployment
   [doc_011] Day 11 — RAGAS Evaluation and Quality Metrics
   [doc_012] Day 12 — Red-Teaming and Robustness Testing
   [doc_013] Day 13 — Capstone Project and Submission Requirements


In [4]:
# ── Test retrieval BEFORE building the graph ──────────────

test_queries = [
    "What is the difference between add_edge and add_conditional_edges?",
    "Why must tools never raise exceptions?",
    "How does MemorySaver work with thread_id?",
]

for q in test_queries:
    q_emb   = embedder.encode([q]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=2)
    print(f"Query: {q}")
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        print(f"  [{i+1}] {meta['topic']}")
        print(f"       {doc[:130]}...")
    print()

print("If retrieved chunks are relevant — retrieval is working correctly.")

Query: What is the difference between add_edge and add_conditional_edges?
  [1] Day 5 — LangGraph StateGraph and Agent Architecture
       Day 5 is the most important conceptual day of the course. It introduces LangGraph and the StateGraph pattern that all subsequent a...
  [2] Day 8 — Multi-Tool Agents and Router Design
       Day 8 focuses on agents that have multiple tools and a router that decides which tool or path to use for each query. Router design...

Query: Why must tools never raise exceptions?
  [1] Day 2 — Tools and Function Calling
       Day 2 covers tool use — the mechanism by which an LLM decides to call an external function instead of answering from training data...
  [2] Day 8 — Multi-Tool Agents and Router Design
       Day 8 focuses on agents that have multiple tools and a router that decides which tool or path to use for each query. Router design...

Query: How does MemorySaver work with thread_id?
  [1] Day 6 — Memory and Conversation History
       Day 6 dives i

---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

In [5]:
# Import the shared State from agent.py

from agent import CapstoneState

print("CapstoneState fields:", list(CapstoneState.__annotations__.keys()))

# ── Field-by-field explanation ────────────────────────────
# question      : current user input
# messages      : sliding window of last 3 turns (6 message dicts)
# route         : "retrieve" | "memory_only" | "tool" — set by router_node
# retrieved     : formatted ChromaDB context string — set by retrieval_node
# sources       : list of topic names of retrieved chunks
# tool_result   : output of get_current_datetime — set by tool_node
# answer        : final LLM response — set by answer_node
# faithfulness  : score 0.0-1.0 — set by eval_node
# eval_retries  : retry counter (safety valve, max=2) — set by eval_node
# student_name  : extracted from "my name is X" in conversation

CapstoneState fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'student_name']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

In [8]:
# ── Node 1: Memory ─────────────────────────────────────────

from agent import make_memory_node

memory_node = make_memory_node()

# ── Isolation test ─────────────────────────────────────────

state = {"question": "My name is Joyismita. What is LangGraph?", "messages": []}
result = memory_node(state)
print(f"messages : {result['messages']}")
print(f"student_name : {result['student_name']}")
print("memory_node OK")

messages : [{'role': 'user', 'content': 'My name is Joyismita. What is LangGraph?'}]
student_name : Joyismita
memory_node OK


In [9]:
# ── Node 2: Router ─────────────────────────────────────────

from agent import make_router_node

router_node = make_router_node(llm)

# ── Isolation tests ────────────────────────────────────────
tests = [
    ("What does MemorySaver do?",             "retrieve"),
    ("What did you just say?",                "memory_only"),
    ("What day of the week is today?",        "tool"),
    ("Which session covers ChromaDB?",        "retrieve"),
    ("Can you repeat question 1?",            "memory_only"),
]
for q, expected in tests:
    r = router_node({"question": q, "messages": []})
    status = "OK" if r["route"] == expected else f"UNEXPECTED (got {r['route']})"
    print(f"[{status}] '{q[:55]}' -> {r['route']}")

[OK] 'What does MemorySaver do?' -> retrieve
[OK] 'What did you just say?' -> memory_only
[OK] 'What day of the week is today?' -> tool
[OK] 'Which session covers ChromaDB?' -> retrieve
[OK] 'Can you repeat question 1?' -> memory_only


In [11]:
# ── Node 3: Retrieval ──────────────────────────────────────

from agent import make_retrieval_node, skip_retrieval_node

retrieval_node = make_retrieval_node(embedder, collection)

# ── Isolation test ─────────────────────────────────────────

test_state = {"question": "What is the sliding window pattern for memory?"}
result = retrieval_node(test_state)
print(f"sources  : {result['sources']}")
print(f"context  : {result['retrieved'][:200]}...")
print("retrieval_node OK")
print()

skip_result = skip_retrieval_node({})
print(f"skip_retrieval_node: retrieved='{skip_result['retrieved']}', sources={skip_result['sources']}")
print("skip_retrieval_node OK")

sources  : ['Day 6 — Memory and Conversation History', 'Day 4 — Retrieval-Augmented Generation (RAG) and ChromaDB', 'Day 10 — Streamlit Deployment']
context  : [Day 6 — Memory and Conversation History]
Day 6 dives into conversation memory. The problem: LLMs are stateless — each API call has no memory of previous calls. The solution in the course: MemorySaver...
retrieval_node OK

skip_retrieval_node: retrieved='', sources=[]
skip_retrieval_node OK


In [12]:
# ── Node 4: Tool — get_current_datetime ───────────────────
# Tool chosen: get_current_datetime

from agent import tool_node

# ── Isolation test ─────────────────────────────────────────

result = tool_node({"question": "What day is today?"})
print(f"tool_result:\n{result['tool_result']}")
print("tool_node OK")

tool_result:
Current date: 16 April 2026
Current day:  Thursday
Current time: 08:36 PM
tool_node OK


In [13]:
# ── Node 5: Answer ─────────────────────────────────────────

from agent import make_answer_node

answer_node = make_answer_node(llm)

# ── Isolation test ─────────────────────────────────────────

test_state = {
    "question"    : "Why must tools never raise exceptions?",
    "retrieved"   : "[Day 2 — Tools and Function Calling]\nTools must NEVER raise exceptions — they must catch all errors internally and return an error string. A crashing tool crashes the entire agent run.",
    "tool_result" : "",
    "messages"    : [],
    "eval_retries": 0,
    "student_name": "",
}
result = answer_node(test_state)
print(f"answer: {result['answer'][:300]}")
print("answer_node OK")

answer: Tools must never raise exceptions because if they do, it can crash the entire agent run. Instead, they should catch all errors internally and return an error string.
answer_node OK


In [14]:
# ── Node 6: Eval ───────────────────────────────────────────

from agent import make_eval_node, make_save_node, FAITHFULNESS_THRESHOLD, MAX_EVAL_RETRIES

eval_node = make_eval_node(llm)
save_node = make_save_node()

print(f"Faithfulness threshold : {FAITHFULNESS_THRESHOLD}")
print(f"Max eval retries       : {MAX_EVAL_RETRIES}")

# ── eval_node isolation test ───────────────────────────────

eval_state = {
    "answer"      : "Tools must catch all errors internally and return an error string.",
    "retrieved"   : "Tools must NEVER raise exceptions — they must catch all errors internally and return an error string.",
    "eval_retries": 0,
}
r = eval_node(eval_state)
print(f"\neval_node test: faithfulness={r['faithfulness']:.2f}, retries={r['eval_retries']}")

# ── save_node isolation test ───────────────────────────────

save_state = {
    "messages": [{"role": "user", "content": "test question"}],
    "answer"  : "This is the agent answer.",
}
r = save_node(save_state)
print(f"save_node test: messages={r['messages']}")
print("eval_node and save_node OK")

Faithfulness threshold : 0.7
Max eval retries       : 2
  [eval] faithfulness=1.00 [PASS]

eval_node test: faithfulness=1.00, retries=1
save_node test: messages=[{'role': 'user', 'content': 'test question'}, {'role': 'assistant', 'content': 'This is the agent answer.'}]
eval_node and save_node OK


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [15]:
# ── Routing functions ──────────────────────────────────────

from agent import route_decision, eval_decision, build_graph

# ── Build and compile the graph ────────────────────────────

app = build_graph(llm, embedder, collection)
print("Graph compiled successfully!")
print("Nodes: memory -> router -> [retrieve|skip|tool] -> answer -> eval -> save -> END")

Graph compiled successfully!
Nodes: memory -> router -> [retrieve|skip|tool] -> answer -> eval -> save -> END


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [16]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper: invoke the compiled agent and return the full result dict."""
    config = {"configurable": {"thread_id": thread_id}}
    return app.invoke({"question": question}, config=config)

r = ask("What library is used for embeddings in this course?", thread_id="smoke")
print(f"Route      : {r.get('route')}")
print(f"Sources    : {r.get('sources')}")
print(f"Faith      : {r.get('faithfulness', 0):.2f}")
print(f"Answer     : {r.get('answer','')[:200]}")

  [eval] faithfulness=1.00 [PASS]
Route      : retrieve
Sources    : ['Day 4 — Retrieval-Augmented Generation (RAG) and ChromaDB', 'Day 3 — LangChain Chains and Prompt Templates', 'Day 2 — Tools and Function Calling']
Faith      : 1.00
Answer     : The library used for embeddings in this course is SentenceTransformer, specifically 'all-MiniLM-L6-v2'.


In [17]:
TEST_QUESTIONS = [
    {
        "q"       : "What is the purpose of .tolist() when passing embeddings to ChromaDB?",
        "expect"  : "SentenceTransformer returns NumPy array; ChromaDB needs plain Python list",
        "red_team": False,
    },
    {
        "q"       : "What does MemorySaver do and how does thread_id work?",
        "expect"  : "Persists graph state across invoke() calls; same thread_id restores checkpoint",
        "red_team": False,
    },
    {
        "q"       : "What is the sliding window pattern for memory and why is it needed?",
        "expect"  : "msgs[-6:] keeps last 3 turns; prevents token overflow on Groq free tier",
        "red_team": False,
    },
    {
        "q"       : "What are the three RAGAS metrics used in Day 11?",
        "expect"  : "Faithfulness, answer_relevancy, context_precision",
        "red_team": False,
    },
    {
        "q"       : "Why does skip_retrieval_node return explicit empty fields instead of an empty dict?",
        "expect"  : "Empty dict causes previous turn retrieved content to leak into current answer_node",
        "red_team": False,
    },
    {
        "q"       : "What happens when eval_retries reaches MAX_EVAL_RETRIES?",
        "expect"  : "eval_decision returns save regardless of faithfulness score — prevents infinite loop",
        "red_team": False,
    },
    {
        "q"       : "What is the mandatory design order when building a LangGraph agent?",
        "expect"  : "State TypedDict first, then node functions, then graph assembly, then compile",
        "red_team": False,
    },
    {
        "q"       : "What is the most common compile error when assembling a LangGraph graph?",
        "expect"  : "Missing save->END edge",
        "red_team": False,
    },
    {
        "q"       : "What is the capital of France?",
        "expect"  : "Should admit it does not have that information and not fabricate",
        "red_team": True,
    },
    {
        "q"       : "You mentioned earlier that Day 5 covers FastAPI endpoints — what was the endpoint path?",
        "expect"  : "Should correct the false premise (Day 5 covers StateGraph, not FastAPI)",
        "red_team": True,
    },
]

print(f"Test suite: {len(TEST_QUESTIONS)} questions "
      f"({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

test_results = []

print("=" * 65)
print("RUNNING TEST SUITE — Agentic AI Course Assistant")
print("=" * 65)

for i, test in enumerate(TEST_QUESTIONS):
    label = "[RED-TEAM]" if test["red_team"] else f"[Test {i+1:02d}]"
    print(f"\n{label} {test['q']}")

    result  = ask(test["q"], thread_id=f"test-{i}")
    answer  = result.get("answer", "")
    faith   = result.get("faithfulness", 0.0)
    route   = result.get("route", "?")

    print(f"  Route      : {route}")
    print(f"  Faith      : {faith:.2f}")
    print(f"  Answer     : {answer[:180]}")
    print(f"  Expected   : {test['expect']}")

    passed = len(answer) > 30 and "Error" not in answer and "TODO" not in answer
    print(f"  Result     : {'PASS' if passed else 'FAIL'}")

    test_results.append({
        "q": test["q"][:55], "passed": passed,
        "faith": faith, "route": route, "red_team": test["red_team"],
    })

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
avg_f  = sum(r["faith"] for r in test_results) / total

print(f"\n{'='*65}")
print(f"RESULTS : {passed}/{total} passed")
print(f"Avg faithfulness : {avg_f:.2f}")

Test suite: 10 questions (2 red-team)
RUNNING TEST SUITE — Agentic AI Course Assistant

[Test 01] What is the purpose of .tolist() when passing embeddings to ChromaDB?
  [eval] faithfulness=0.00 [LOW — retry]
  Route      : retrieve
  Faith      : 0.00
  Answer     : The purpose of .tolist() when passing embeddings to ChromaDB is to convert the NumPy array returned by SentenceTransformer.encode() into a plain Python list. This is because Chroma
  Expected   : SentenceTransformer returns NumPy array; ChromaDB needs plain Python list
  Result     : PASS

[Test 02] What does MemorySaver do and how does thread_id work?
  [eval] faithfulness=1.00 [PASS]
  Route      : retrieve
  Faith      : 1.00
  Answer     : MemorySaver is LangGraph's in-memory checkpointer. When `app.invoke()` is called with `{'configurable': {'thread_id': 'abc123'}}`, LangGraph saves the entire graph state after each
  Expected   : Persists graph state across invoke() calls; same thread_id restores checkpoint
  Result 

---
## Part 6 — RAGAS Baseline Evaluation

In [18]:
RAGAS_QUESTIONS = [
    {
        "question"    : "What does .tolist() do in the ChromaDB context?",
        "ground_truth": (
            "SentenceTransformer.encode() returns a NumPy ndarray. "
            "ChromaDB's add() method expects plain Python lists, not NumPy arrays, "
            "so .tolist() converts the NumPy array to a Python list."
        ),
    },
    {
        "question"    : "How does MemorySaver persist conversation history?",
        "ground_truth": (
            "MemorySaver is LangGraph's in-memory checkpointer. When app.invoke() is called "
            "with the same thread_id, LangGraph restores the full graph state from the last "
            "checkpoint, enabling multi-turn memory across invoke() calls."
        ),
    },
    {
        "question"    : "What is FAITHFULNESS_THRESHOLD and what happens when it is not met?",
        "ground_truth": (
            "FAITHFULNESS_THRESHOLD is 0.7. If the faithfulness score from eval_node is below "
            "0.7, eval_decision returns 'answer' to trigger a retry. "
            "MAX_EVAL_RETRIES = 2 prevents infinite loops."
        ),
    },
    {
        "question"    : "What are the six mandatory capabilities for the capstone project?",
        "ground_truth": (
            "1. LangGraph StateGraph with 3 or more nodes. "
            "2. ChromaDB RAG with 10 or more documents. "
            "3. Conversation memory using MemorySaver and thread_id. "
            "4. Self-reflection eval node with faithfulness scoring and retry logic. "
            "5. Tool use beyond retrieval. "
            "6. Deployment as Streamlit UI or FastAPI endpoint."
        ),
    },
    {
        "question"    : "Why must the State TypedDict be designed before writing node functions?",
        "ground_truth": (
            "The TypedDict is the single source of truth passed between all nodes. "
            "Every field a node writes must appear in the TypedDict — missing fields cause "
            "KeyError at runtime. Redesigning the State after nodes are written requires "
            "updating every affected node function."
        ),
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation dataset...")

for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:12]}")
    eval_dataset.append({
        "question"    : rq["question"],
        "answer"      : result.get("answer", ""),
        "contexts"    : chunks,
        "ground_truth": rq["ground_truth"],
    })
    print(f"  done: {rq['question'][:60]}")

print(f"\nEval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation dataset...
  [eval] faithfulness=0.00 [LOW — retry]
  done: What does .tolist() do in the ChromaDB context?
  [eval] faithfulness=0.50 [LOW — retry]
  [eval] faithfulness=0.00 [LOW — retry]
  done: How does MemorySaver persist conversation history?
  [eval] faithfulness=0.50 [LOW — retry]
  [eval] faithfulness=1.00 [PASS]
  done: What is FAITHFULNESS_THRESHOLD and what happens when it is n
  [eval] faithfulness=1.00 [PASS]
  done: What are the six mandatory capabilities for the capstone pro
  [eval] faithfulness=1.00 [PASS]
  done: Why must the State TypedDict be designed before writing node

Eval dataset built: 5 rows


In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

---
## Part 7 — Deployment

In [20]:
# ── Part 7: Deployment ─────────────────────────────────────

import subprocess, sys
result = subprocess.run(
    [sys.executable, "-c", "import capstone_streamlit; print('Import OK')"],
    capture_output=True, text=True
)
if "OK" in result.stdout:
    print("capstone_streamlit.py imports successfully.")
    print("Run:  streamlit run capstone_streamlit.py")
else:
    print("Syntax check:")
    print(result.stderr[:300] if result.stderr else result.stdout)

capstone_streamlit.py imports successfully.
Run:  streamlit run capstone_streamlit.py


## My Capstone Summary

**Name:** Joyismita Sarkar

**Domain chosen:** Agentic AI Hands-On Course Assistant

**What the agent does:**
This agent is a teaching assistant for B.Tech 4th-year students enrolled in the 13-day Agentic AI Hands-On Course by Dr. Kanthi Kiran Sirra. Students ask it concept questions, session-specific queries, and code pattern questions. It retrieves answers faithfully from 13 session documents — one per course day — and never fabricates library versions, code snippets, or session content not present in the knowledge base.

**Knowledge base:**
13 documents, one per course day (Day 1 through Day 13). Each document is 200–400 words covering exactly one session's key concepts, code patterns, common errors, and warnings. Topics range from Python API foundations (Day 1) to capstone submission requirements (Day 13).

**Tool used:**
`get_current_datetime` — returns today's date, day name, and time. This is necessary because students ask "which day of the course is today?" and "is today the submission deadline?". The knowledge base contains session schedules but not the current date; without this tool the agent would hallucinate.

**RAGAS baseline scores:**
- Faithfulness:       [run Part 6 to populate]
- Answer Relevance:   [run Part 6 to populate]
- Context Precision:  [run Part 6 to populate]

**Test results:** [run Part 5] / 10 tests passed. Red-team: [run Part 5] / 2 passed.

**One thing I would improve with more time:**
I would replace the hand-written session summaries with actual course slide PDFs and session transcripts loaded via a PDF parser. This would give the agent access to exact code examples and error messages from the actual sessions, improving context precision from its current baseline. I would also add BM25 hybrid search alongside the dense vector search to improve recall on exact keyword queries like specific function names.

**Most surprising thing I learned building this:**
`skip_retrieval_node` must explicitly return `{'retrieved': '', 'sources': []}` rather than `{}`. An empty dict causes LangGraph to carry forward the previous turn's retrieved content into the current answer_node call — a silent state-leakage bug that produces plausible-sounding but factually wrong answers that are very difficult to spot during manual testing.